In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import warnings
from scipy.integrate import solve_ivp
from numba import njit
from itertools import product
 
warnings.filterwarnings("ignore")
 
@njit(cache=True, fastmath=True)
def Rossler_rhs(X, a, b, c, N, L, eps):
    x, y, z = np.split(X, 3)
    dx = -y -z
    dy =  x + a*y + eps * L @ y
    dz =  b + z*x - c*z
    return np.concatenate((dx, dy, dz))

@njit(cache=True, fastmath=True)
def variational_rhs(Y, a, b, c, N, L, eps, k):
    # y = [x (3N,), Q (3N, k) flattened]
    X = Y[:3*N]
    Q = Y[3*N:].reshape(3*N, k)
    dx = Rossler_rhs(X, a, b, c, N, L, eps)

    J = jacobian_ross(X, a, c, eps, L, N)
    dQ = J@Q
    return np.concatenate((dx, dQ.ravel()))

@njit(cache=True, fastmath=True)
def jacobian_ross(X, a, c, eps, L, N):
    x = X[:N]; z = X[2*N:]
    J = np.zeros((3*N, 3*N))
    I = np.identity(N)
    J[0:N,     N:2*N]   = -I
    J[0:N,     2*N:3*N] = -I
    J[N:2*N,   0:N]     =  I
    J[N:2*N,   N:2*N]   =  np.diag(a) + eps * L
    for i in range(N):
        J[2*N + i, i]         = z[i]
        J[2*N + i, 2*N + i]   = x[i] - c
    return J

@njit(cache=True, fastmath=True)
def rk4_Rossler_step(X, dt, a, b, c, N, L, eps):
    k1 = Rossler_rhs(X, a, b, c, N, L, eps)
    k2 = Rossler_rhs(X + 0.5*dt*k1, a, b, c, N, L, eps)
    k3 = Rossler_rhs(X + 0.5*dt*k2, a, b, c, N, L, eps)
    k4 = Rossler_rhs(X +     dt*k3, a, b, c, N, L, eps)
    return X + dt*(k1 + 2*k2 + 2*k3 + k4)/6

@njit(cache=True, fastmath=True)
def rk4_variational_step(X, dt, a, b, c, N, L, eps, k):
    k1 = variational_rhs(X, a, b, c, N, L, eps, k)
    k2 = variational_rhs(X + 0.5*dt*k1, a, b, c, N, L, eps, k)
    k3 = variational_rhs(X + 0.5*dt*k2, a, b, c, N, L, eps, k)
    k4 = variational_rhs(X +     dt*k3, a, b, c, N, L, eps, k)
    return X + dt*(k1 + 2*k2 + 2*k3 + k4)/6

def largest_lyapunov(a, N, b, c , L, eps, x0, t_transient, t_total, renorm_dt, k, rng):
    """Benettin's algorithm. Returns the k largest Lyapunov exponents."""
    rng = np.random.default_rng()

    # 1) Burn off the transient so we are actually on the attractor.
    dt=0.01
    X = rk4_Rossler_step(x0, dt, a, b, c, N, L, eps)
    for i in range (int(t_transient/dt)):
        X = rk4_Rossler_step(X, dt, a, b, c, N, L, eps)

    # 2) Start with an orthonormal set of tangent vectors.
    Q, _ = np.linalg.qr(rng.standard_normal((3*N, k)))

    # 3) Evolve + renormalize.
    n_steps = int(t_total / renorm_dt)
    log_sums = np.zeros(k)
    for _ in range(n_steps):
        y0 = np.concatenate((X, Q.ravel()))
        Y = rk4_variational_step(y0, dt, a, b, c, N, L, eps, k)
        for i in range (int(renorm_dt/dt)):
            Y = rk4_variational_step(Y, dt, a, b, c, N, L, eps, k)

        X = Y[:3*N]
        Q = Y[3*N:].reshape(3*N, k)
        Q, R = np.linalg.qr(Q)    # reorthonormalize
        log_sums += np.log(np.abs(np.diag(R)))


        plt.show()

    return log_sums / (n_steps * renorm_dt) 

def one_job(mc_idx, k_idx, seed_mc, sigma_k, N, b, c, L, eps):
    rng = np.random.default_rng(seed_mc)            # same RNG per MC across all sigmas

    a = 0.1*np.ones(N)
    d_a = rng.random(N)

    a_fin = a+sigma_k/N*d_a
    

    x0 = np.linspace(1, 3*N, 3*N) / 3*N
    x0 = x0 / x0.sum()
    
    t_trans = 16000
    lam_vec =largest_lyapunov(a_fin, N, b, c , L, eps, x0,
                           t_transient=t_trans,
                           t_total=10 * t_trans,
                           renorm_dt=1, k=2, rng=rng)
    lam_0, lam_1 = lam_vec[0], lam_vec[1]
    return mc_idx, k_idx, lam_0, lam_1
 
 

# ------- Default parameters -------------------------------------------
N = 10                              # number of nodes
Nhet = 1000                          # number of heterogeneity steps
sigma = np.linspace(0, 1, Nhet)     # heterogeneity interval                    # growth heterogeneity
Nmc = 10         # Monte Carlo realizations
b = 0.1
c = 14.0
eps = 0.225
L=np.zeros((N,N))
for i in range (N):
    L[i][i]=-2
    L[i][(i+1)%N]=1
    L[i][(i-1)%N]=1



# Parallel Monte Carlo loop
base_seed = np.random.SeedSequence().entropy
mc_seeds = [base_seed + i for i in range(Nmc)]

jobs = list(product(range(Nmc), range(Nhet)))

results = Parallel(n_jobs=-1, backend="loky", batch_size="auto")(
    delayed(one_job)(i, k, mc_seeds[i], sigma[k], N, b, c, L, eps) for i, k in jobs)

LyapExp_run_0 = np.empty((Nmc, Nhet))
LyapExp_run_1 = np.empty((Nmc, Nhet))
for i, k, lam_0, lam_1 in results:
    LyapExp_run_0[i, k], LyapExp_run_1[i, k] = lam_0, lam_1

# Performance analysis 
stabilityimprovement = [] 
stabilizedfraction = [] 


# ------- Plot ----------------------------------------------------------
Nsamps = 10
j = 0

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

#print()
ax = axes[0, 0]
ax.plot(sigma, (LyapExp_run_0[:Nsamps, :]).T,linewidth=2)
ax.plot(sigma, np.zeros(len(sigma)), color='black')
ax.set_xlabel(r"heterogeneity $\sigma$")
ax.set_ylabel(r"$\lambda_{0}$")

ax = axes[0, 1]
ax.plot(sigma, (LyapExp_run_1[:Nsamps, :]).T,linewidth=2)
ax.plot(sigma, np.zeros(len(sigma)), color='black')
ax.set_xlabel(r"heterogeneity $\sigma$")
ax.set_ylabel(r"$\lambda_{1}$")

for ax in axes.flat:
    for item in ([ax.title, ax.xaxis.label, ax.yaxis.label] +
                    ax.get_xticklabels() + ax.get_yticklabels()):
        item.set_fontsize(14)

plt.tight_layout()
plt.show()